# **Practical assignment for Topic 9**

In the assignment, you have to use the dataset `auto-lpkm.tsv` (with quadratic features) given in the lecture and you should reuse all the given code.

Your assignment consists of the following tasks:
<br>
<br>

**Task 0**<br>
Take the SFS implementation from Lecture 8 and run it on the dataset. Note the best found model and its coefficient of determination.
<br>
<br>

**Task 1**<br>
In the function `GA()`, add the following additional implementations:
1. Implement tournament selection. Set tournament size to 4.
2. Implement <a href="https://en.wikipedia.org/wiki/Crossover_(genetic_algorithm)#Uniform_crossover">uniform cross-over</a>. As discussed in the lecture, for each gene there should be 50% probability to go to the first child and other 50% probability to go to the second child. And then the other parent's gene does the opposite.
3. Implement setting random seed to be an argument of the function.
4. Set `num_generations` to 50.

While adding the implementations, do not remove the roulette wheel selection and the single-point cross-over. Instead write the code so that the function has additional arguments using which the caller can choose which selection or cross-over method to use.
<br>
<br>

**Task 2.1**<br>
Run the following four configurations of GA, each 10 times with different random seeds (without any other configuration changes):
* Configuration 1: Roulette wheel selection and single-point cross-over.
* Configuration 2: Roulette wheel selection and uniform cross-over.
* Configuration 3: Tournament selection and single-point cross-over.
* Configuration 4: Tournament selection and uniform cross-over.

From the results, for each configuration, compute the average $R^2$ and its standard deviation. Also, for each configuration, plot all 10 "generation vs. best fittness" curves in the same plot. In total, you will have 4 plots, 10 curves in each.
<br>
<br>

**Task 2.2**<br>
In this task you are free to try any other configuration changes to the GA algorithm implementation in order to obtain better results.
<br>
<br>

**Task 3**<br>
Now write consise analysis of all the results you obtained in this assignment.

In [ ]:
import numpy as np
from sklearn.model_selection import cross_validate

In [ ]:
def GA(X, y, model, feature_names):
    m = X.shape[1]

    # configuration
    pop_size = 10 + 1  # even number plus elite
    num_generations = 30
    crossover_prob = 0.9
    mutation_prob = 1 / m

    np.random.seed(42)

    # matrices to store the current and the next generation
    pop = np.random.choice(a=[False, True], size=(pop_size, m))
    pop_next = np.empty((pop_size, m), dtype=bool)

    # overall best found soluton
    overall_best_chromosome = np.full(m, False)  # the overall best found solution
    overall_best_fitness = np.inf  # fitness of the best found solution

    # for storing statistics
    stats = np.empty((num_generations, 2))

    # GA generations (iterations)
    for generation in range(0, num_generations):

        # evaluation of each individual
        fitness = np.empty(pop_size)
        for i in range(0, pop_size):
            chromosome = pop[i, :]
            if not np.any(chromosome):
                fitness[i] = np.inf
                continue
            X_selected = X[:, chromosome]
            fitness[i] = -cross_validate(model, X_selected, y, cv=10, scoring=["neg_mean_squared_error"])[
                'test_neg_mean_squared_error'].mean()

        # find the best individual of the generation
        best_individual = np.argmin(fitness)
        best_fitness = fitness[best_individual]
        best_chromosome = pop[best_individual, :]
        # save best fitness of the generation to stats
        stats[generation] = [np.ma.masked_invalid(fitness).mean(), best_fitness]
        # if better than the overall best, remember it
        if best_fitness < overall_best_fitness:
            overall_best_fitness = best_fitness
            overall_best_chromosome = best_chromosome.copy()

        # print best result in the current generation
        print(f'Generation {generation}: Best fitness = {fitness[best_individual]}')
        if feature_names.shape[0] == m:
            print(''.join(['1' if bit else '0' for bit in pop[best_individual, :]]) + ': ' + '\t'.join(
                feature_names[pop[best_individual, :]]))
        else:
            print(''.join(['1' if bit else '0' for bit in pop[best_individual, :]]))

        # no need to run the rest of the code for the last generation
        if generation >= num_generations - 1:
            break

        # some preparations for roulette wheel selection
        inverted_fitness = 1 / fitness
        roulette_probs = np.zeros(pop_size)
        roulette_probs[0] = inverted_fitness[0] / np.sum(inverted_fitness)
        for i in range(1, pop_size):
            roulette_probs[i] = roulette_probs[i - 1] + inverted_fitness[i] / np.sum(inverted_fitness)

        # loop through population size with step 2 so that we fully create the next population
        for i in range(0, pop_size - 1, 2):  # -1 because of the elitism
            # roulette wheel selection (finding the first sum of probs that is larger than a random number)
            parent1 = np.argmax(roulette_probs >= np.random.rand())
            parent2 = np.argmax(roulette_probs >= np.random.rand())

            # single-point crossover
            if crossover_prob > np.random.rand():
                crossover_pos = np.random.randint(1, high=m)
                pop_next[i, :] = np.hstack([pop[parent1, 0:crossover_pos], pop[parent2, crossover_pos:]])
                pop_next[i + 1, :] = np.hstack([pop[parent2, 0:crossover_pos], pop[parent1, crossover_pos:]])
            else:
                pop_next[i, :] = pop[parent1, :]
                pop_next[i + 1, :] = pop[parent2, :]

        # elitist
        pop_next[-1, :] = best_chromosome

        # mutation
        mutate = np.random.rand(pop_size, m) <= mutation_prob  # mutation mask for all individuals
        mutate[-1, :] = False  # do not mutate elite
        np.logical_not(pop_next, where=mutate, out=pop_next)

        # the next population becomes the current one
        pop = pop_next.copy()

    # print the overall best result
    print()
    print('Overall best fitness =', overall_best_fitness)
    if feature_names.shape[0] == m:
        print(''.join(['1' if bit else '0' for bit in overall_best_chromosome]) + ': ' + '\t'.join(
            feature_names[overall_best_chromosome]))
    else:
        print(''.join(['1' if bit else '0' for bit in overall_best_chromosome]))

    # in addition, compute coefficient of determination
    X_selected = X[:, overall_best_chromosome]
    overall_best_R2 = cross_validate(model, X_selected, y, cv=10, scoring=["r2"])['test_r2'].mean()
    print('\nOverall best R2: %.3f' % overall_best_R2)

    return stats, overall_best_R2

In [ ]:
def SFS(X, y, model, feature_names):
    m = X.shape[1]

    current_bits = np.full(m, False)  # the current remembered solution

    overall_best_bits = np.full(m, False)  # the overall best found solution
    overall_best_mse = np.inf  # evaluation of the best found solution

    # SFS iterations (as many iteration as there are features)
    for iter in range(1, m + 1):
        best_mse = np.inf  # best evaluation in the current iteration
        best_j = -1  # best feature in the current iteration

        # going through all features
        for j in range(0, m):
            if current_bits[j]:  # if the feature is already included, we do nothing
                continue
            current_bits[j] = True  # inclusion of the feature

            X_selected = X[:, current_bits]  # data with the selected features
            MSE = -cross_validate(model, X_selected, y, cv=10, scoring=["neg_mean_squared_error"])[
                'test_neg_mean_squared_error'].mean()
            # if better, remember it
            if MSE < best_mse:
                best_mse = MSE
                best_j = j

            current_bits[j] = False  # remove it to get back to original vector

        current_bits[best_j] = True  # inclusion of the best found feature

        # print best result in the current iteration
        print(f'Iteration {iter}: MSE = {best_mse}')
        print(''.join(['1' if bit else '0' for bit in current_bits]) + ': ' + '\t'.join(feature_names[current_bits]))
        print()

        # if better than the overall best, remember it
        if best_mse < overall_best_mse:
            overall_best_mse = best_mse
            overall_best_bits = current_bits.copy()

    # print the overall best result
    print('Overall best MSE =', overall_best_mse)
    print(''.join(['1' if bit else '0' for bit in overall_best_bits]) + ': ' + '\t'.join(
        feature_names[overall_best_bits]))

    # in addition, compute coefficient of determination
    X_selected = X[:, overall_best_bits]
    R2 = cross_validate(model, X_selected, y, cv=10, scoring=["r2"])['test_r2'].mean()
    print('\nOverall best R2: %.3f' % R2)

---
*While doing the tasks you are allowed to use only those function libraries that were already used in the lectures so far.*

**After the tasks are done, submit this file. Do not clear it's output - all print-outs and figures should be left in the file. All your code should be fully runnable in the order of appearance and it should produce exactly the same output that is present in your print-outs and figures you left in the file.**